# PitchVision — pipeline demo

This notebook runs the PitchVision spotting pipeline on one held-out match and presents the output as a
searchable timeline. It uses the seed-0, W = 7 checkpoint from the report
(`results/checkpoints/v3_ckpt_W7_seed0.pt`) and the packaged `scripts/spot_video.py`. Neither the model nor its
settings are modified for the demonstration.

| Cell | Purpose |
|---|---|
| 1 · Setup | Mounts Drive, locates the code package and the match video, reports the compute device |
| 2 · Helpers | Functions to run the pipeline, print the timeline, filter by event type and compare with the labels |
| 3 · Warm-up run | Runs the pipeline once, reports the run time and stores a copy of the outputs in Drive |
| 4 · Run the pipeline | Runs the pipeline on the first `SECONDS` seconds of the match |
| 5 · Timeline | Prints the detected events in time order |
| 6 · Filter by event type | Lists the detected events of one type |
| 7 · Compare with ground-truth labels | Checks the listed events against the professional annotations |
| 8 · Load the stored run | Loads the outputs stored by the warm-up run in place of a fresh run |

**Notes on the output.** Scores are the model's raw sigmoid outputs; because event classes are weighted 15x during
training they are not calibrated probabilities. A detected event counts as matching an annotation when an
annotated event of the same type lies within 5 seconds of it.

In [ ]:
#@title 1 · Setup { display-mode: "form" }
import json, os, shutil, subprocess, sys, time, zipfile
import numpy as np
from google.colab import drive
drive.mount('/content/drive')

# --- Inputs (already stored in Google Drive) ----------------------------------------------------
ZIP       = '/content/drive/MyDrive/pitchvision_final_submission.zip'     # code package, includes the checkpoints
MATCH_DIR = ('/content/drive/MyDrive/SoccerNet_Project/extracted/test/england_efl/2019-2020/'
             '2019-10-01 - Stoke City - Huddersfield Town')               # held-out match, never used for training
VIDEO     = f'{MATCH_DIR}/224p.mp4'
LABELS    = f'{MATCH_DIR}/Labels-ball.json'

# --- Settings -----------------------------------------------------------------------------------
SECONDS      = 600                                                        # seconds of video per run (600 = first 10 minutes)
OUT          = '/content/demo'                                            # working directory for pipeline outputs
FALLBACK_DIR = '/content/drive/MyDrive/PitchVision_Demo_fallback'         # stored copy of a finished run (kept in Drive)

for path in (ZIP, VIDEO):
    if not os.path.exists(path):
        raise FileNotFoundError(path)

import torch
GPU = torch.cuda.is_available()
print('Device :', torch.cuda.get_device_name(0) if GPU else 'CPU (no GPU available)')
if not GPU:
    SECONDS = min(SECONDS, 120)    # feature extraction is roughly ten times slower on CPU, so process a shorter stretch
    print(f'         processing {SECONDS} s per run')

shutil.rmtree('/content/pv', ignore_errors=True)
zipfile.ZipFile(ZIP).extractall('/content/pv')
PKG  = '/content/pv/pitchvision'
CKPT = f'{PKG}/results/checkpoints/v3_ckpt_W7_seed0.pt'
assert os.path.exists(CKPT), 'checkpoint not found in the code package'
sys.path.insert(0, PKG)

print('Code   :', PKG)
print('Video  :', os.path.basename(MATCH_DIR), f'({os.path.getsize(VIDEO) / 1e6:.0f} MB)')
print('Labels :', 'available' if os.path.exists(LABELS) else 'not found')

In [ ]:
#@title 2 · Helpers { display-mode: "form" }
from collections import Counter
from pitchvision.data import CLASSES, build_labels
from pitchvision.metrics import peaks

# Display names for the twelve event classes.
NICE = {'PASS': 'Pass', 'DRIVE': 'Drive (ball carried)', 'HEADER': 'Header', 'HIGH PASS': 'High pass',
        'OUT': 'Ball out of play', 'CROSS': 'Cross', 'THROW IN': 'Throw-in', 'SHOT': 'Shot',
        'BALL PLAYER BLOCK': 'Block', 'PLAYER SUCCESSFUL TACKLE': 'Tackle', 'FREE KICK': 'Free kick', 'GOAL': 'Goal'}
TL = None    # timeline of the most recent run: {'W': ..., 'threshold': ..., 'events': [...]}

def _bar(x):
    """Ten-character text bar for a score in [0, 1]."""
    n = int(round(min(max(x, 0), 1) * 10))
    return '#' * n + '.' * (10 - n)

def _print_rows(rows):
    print(f"{'TIME':<8}{'EVENT':<24}SCORE")
    for e in rows:
        print(f"{e['mmss']:<8}{NICE[e['class']]:<24}{_bar(e['score'])}  {e['score']:.2f}")

def run_demo(seconds=None):
    """Run scripts/spot_video.py on the match and return the parsed timeline."""
    seconds = seconds or SECONDS
    t0 = time.time()
    r = subprocess.run([sys.executable, f'{PKG}/scripts/spot_video.py', '--video', VIDEO, '--ckpt', CKPT,
                        '--out', OUT, '--max_seconds', str(seconds)], capture_output=True, text=True)
    if r.returncode:
        raise RuntimeError('spot_video.py failed:\n' + r.stderr[-1200:])
    tl = json.load(open(f'{OUT}/timeline.json'))
    print(f"Processed {seconds // 60}:{seconds % 60:02d} of video in {time.time() - t0:.0f} s "
          f"-> {len(tl['events'])} events")
    return tl

def show_timeline(limit=12):
    """Print the first `limit` events in time order with a summary of the most common types."""
    ev = TL['events']
    top = Counter(e['class'] for e in ev).most_common(4)
    print(f"{len(ev)} events.  Most common: " + ',  '.join(f"{NICE[k]} {v}" for k, v in top))
    print()
    _print_rows(ev[:limit])
    print(f"\n... {max(len(ev) - limit, 0)} more.   (score = raw model output, not a calibrated probability)")

def find(cls, k=8, quiet=False):
    """List the detected events of one type.

    If no event of that type reaches the display threshold, the k highest-scoring local maxima of that
    class are listed instead, and the header says so."""
    cls = cls.upper()
    ev = [e for e in TL['events'] if e['class'] == cls]
    if ev:
        rows = ev[:k]
        head = f"{len(ev)} {NICE[cls].lower()} events. First {len(rows)}:"
    else:
        P = np.load(f'{OUT}/scores.npy'); pk = peaks(P); h = TL['W'] // 2; c = CLASSES.index(cls)
        idx = [i for i in np.argsort(-pk[:, c])[:k] if pk[i, c] >= 0]
        rows = sorted(({'time_s': int(i + h), 'mmss': f'{(i + h) // 60:02d}:{(i + h) % 60:02d}',
                        'class': cls, 'score': round(float(P[i, c]), 2)} for i in idx), key=lambda e: e['time_s'])
        head = (f"No {NICE[cls].lower()} event reached the display threshold ({TL['threshold']}); "
                f"the {len(rows)} highest-scoring candidates:")
    if not quiet:
        print(head); print()
        _print_rows(rows)
    return rows

def check(cls, rows=None, tol=5):
    """Compare listed events with the professional annotations.

    An event matches when an annotated event of the same type lies within `tol` seconds of it."""
    if not os.path.exists(LABELS):
        print('No annotation file available.'); return
    cls = cls.upper(); rows = rows or find(cls, quiet=True)
    Y, _, _ = build_labels(LABELS, SECONDS + 1, CLASSES)
    g = np.where(Y[:, CLASSES.index(cls)] > 0)[0]
    hits = [bool(len(g)) and bool(np.min(np.abs(g - e['time_s'])) <= tol) for e in rows]
    print(f"{sum(hits)} of {len(rows)} listed {NICE[cls].lower()} events match an annotated event within {tol} s.")
    print(f"The annotations contain {len(g)} {NICE[cls].lower()} events in these {SECONDS // 60} minutes.\n")
    for e, ok in zip(rows, hits):
        print(f"  {e['mmss']}   {'matches an annotation' if ok else 'no matching annotation'}")

def save_fallback():
    """Store a copy of the current outputs in Drive."""
    shutil.rmtree(FALLBACK_DIR, ignore_errors=True)
    shutil.copytree(OUT, FALLBACK_DIR)

def use_fallback():
    """Load the stored outputs of the warm-up run."""
    global TL, OUT
    OUT = FALLBACK_DIR
    TL = json.load(open(f'{OUT}/timeline.json'))
    print(f"Loaded stored run: {len(TL['events'])} events.")

In [ ]:
#@title 3 · Warm-up run { display-mode: "form" }
t0 = time.time()
TL = run_demo()
elapsed = time.time() - t0
save_fallback()
print(f'Run time    : {elapsed:.0f} s')
print(f'Stored copy : {FALLBACK_DIR}')
print()
show_timeline(6)

In [ ]:
#@title 4 · Run the pipeline { display-mode: "form" }
TL = run_demo()

In [ ]:
#@title 5 · Timeline { display-mode: "form" }
show_timeline()

In [ ]:
#@title 6 · Filter by event type { display-mode: "form" }
EVENT = "CROSS"   #@param ["CROSS", "HEADER", "HIGH PASS", "THROW IN", "PASS", "DRIVE", "OUT"]
rows = find(EVENT)

In [ ]:
#@title 7 · Compare with ground-truth labels { display-mode: "form" }
check(EVENT, rows)

In [ ]:
#@title 8 · Load the stored run { display-mode: "form" }
use_fallback()
show_timeline()